In [ ]:
 import pandas as pd
 import numpy as np
 import matplotlib.pyplot as plt
 import seaborn as sns

In [ ]:
import regex as re
import nltk
import textblob
import tweepy
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('stopwords')

from nltk.stem import PorterStemmer

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# mention of Brandwidth Analytics, a social media tools tool

twitter's tweepy API  
1) apply for a developer account   
> you will need a twitter user account  
2) if prompted: choose a personal or individual account rather than a business one  
3) when subbmitting intended use, answer should be sufficiently descriptive  
4) create an app so you can register it at https://developer.twitter.com/en/apps  
5) after you set up, generate an access token and secret  

In [ ]:
consumerKey='u8dtgclFiERG5ax27YIx18IMv'
consumerSecret='pBBOAqEvMKPYuUvOZBiT6AAaFdme2H4mcKtaFjhyZBmN7i6tdT'
accessToken='1759749853410717696-c4T8YDG8o9lBNcITPBKUFOejsHdc9W'
accessTokenSecret='3FfwwrUlH7pVKdnVktp1OpRXzCQa0ekT6eQglM1V04exo'
auth = tweepy.OAuthHandler(consumerKey,consumerSecret)
auth.set_access_token(accessToken,accessTokenSecret)
api = tweepy.API(auth)

In [ ]:
searchTerm=input(f"Enter Keyword/Tag to search about: ")
NumOfTerms=int(input(f"Ender how many tweets to search: "))

tweets =[]
tweetText=[]
tweets = tweepy.Cursor(api.search_tweets,q=searchTerm+" -filter:retweets",lang="en").items(NumOfTerms)


Enter Keyword/Tag to search about: Chicago Bears
Ender how many tweets to search: 79


In [ ]:
tweet_list = [tweet.text for tweet in tweets]
tweet_df = pd.DataFrame(tweet_list)
tweet_df

Forbidden: 403 Forbidden
453 - You currently have access to a subset of X API V2 endpoints and limited v1.1 endpoints (e.g. media post, oauth) only. If you need access to this endpoint, you may need a different access level. You can learn more here: https://developer.x.com/en/portal/product

In [ ]:
#perhaps this can be vectorized better
"""
def clean_data(text):
  return ' '.join(re.sub("(@[a-zA-Z0-9]+)|([^0-9A-Za-z])|https://[\w.]+/[\w]+)"," ",text).split())
"""

<>:2: SyntaxWarning: invalid escape sequence '\w'
<>:2: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipython-input-1141106178.py:2: SyntaxWarning: invalid escape sequence '\w'
  return ' '.join(re.sub("(@[a-zA-Z0-9]+)|([^0-9A-Za-z])|https://[\w.]+/[\w]+)"," ",text).split())


In [ ]:
#tweet_df['Cleaned_data']=tweet_df[0].map(clean_data)# it was .apply and lambda x: in the vid

In [ ]:
"""
def drop_numbers(list_text):
  list_text_new=[]
  for i in list_text:
    if not re.search('\d',i):
      list_text_new.append(i)
  return ''.join(list_text_new)

tweet_df['Cleaned_data']=tweet_df['Cleaned_data'].map(drop_numbers)
"""


In [ ]:
"""
def lower_case(text):
  text_words = word_tokenize(text)
  text_words_lower = [i.lower() for i in text_words]
  return ' '.join(text_words_lower)

tweet_df['Cleaned_data']=tweet_df['Cleaned_data'].map(lower_case)
"""

In [ ]:
"""lemmatizer=WordNetLemmatizer()
def lemmatize(text):
  text_tokens = word_tokenize(text)
  lemmi = [lemmatizer.lemmatize(word) for word in text_tokens]
  return ' '.join(lemmi)
tweet_df['Cleaned_data']=tweet_df['Cleaned_data'].map(lemmatize)
"""

In [ ]:
""
def remove_stopwords(text):
  text_tokens = word_tokenize(text)
  tokens = [word for word in text_tokens if not word in set(stopwords.words('english'))]
  tokens_text = ' '.join(tokens)
  return tokens_text

tweet_df['Cleaned_data']=tweet_df['Cleaned_data'].map(remove_stopwords)
"""

In [ ]:
lemmatizer=WordNetLemmatizer()
def lemmatize(text):
  text_tokens = text.split()
  for pos in ['a','v','r','n','s']:
    lemmed=list(map(lambda x: WordNetLemmatizer().lemmatize(x,pos=pos),text_tokens))
  return ' '.join(text_tokens)
def remove_stopwords(text):
  text_tokens = text.split()
  no_stopwords = list(filter(lambda x: x not in stopwords.words('english'),text_tokens))
  return ' '.join(no_stopwords)
porterstemmer=PorterStemmer()
def porter_stemmer(text):
  text_tokens = text.split()
  stemmed_words = list(map(lambda x: porterstemmer.stem(x),text_tokens))
  return ' '.join(stemmed_words)

def process_data(text_data_vector):
  text_data_vector= text_data_vector.str.replace(to_replace=r"(@[a-zA-Z0-9]+)|([^0-9A-Za-z])|(https://[\w.]+/[\w]+)",value=" ",regex=True)
  text_data_vector= text_data_vector.replace(to_replace=r"\d",value="",regex=True)
  text_data_vector= text_data_vector.str.lower()
  text_data_vector= text_data_vector.map(lemmatize)
  text_data_vector= text_data_vector.map(remove_stopwords)
  text_data_vector= text_data_vector.map(porter_stemmer)
  return text_data_vector



In [ ]:
tweet_df['Cleaned_data']=process_data(tweet_df[0])

In [ ]:
def get_polarity(text):
  textblob = TextBlob(str(text))
  pol = textblob.sentiment.polarity
  return pol

def interpret_polarity(df,polarity_column):
  conditions = {
          'Neutral': lambda pol: (pol==0),
          'Weak Positive': lambda pol: (pol>0 & pol<=0.3),
           "Positive": lambda pol: (pol>0.3 & pol<=0.6),
           'Strong Positive':  lambda pol: (pol>0.6 & pol<=1),
           'Weak Negative': lambda pol: (pol>-0.3 & pol<=0),
           'Negative': lambda pol: (pol>-0.6 and pol<=-0.3),
           'Strong Negative':  lambda pol: (pol>=-1 & pol<=-0.6),
          }
  df['sentiment']=np.nan
  for label,mask in conditions.items():
    msk=mask(df[polarity_column])
    df.loc[msk,'sentiment']=label
  return df


tweet_df['polarity']=tweet_df['Cleaned_data'].map(get_polarity)
tweet_df['sentiment']=tweet_df['polarity'].map(interpret_polarity)




In [ ]:
tweets_df['sentiment'].value_counts()

In [ ]:
print(f"Average Tweet Polarity from -1 to 1: {round(tweets_df['polarity'].mean(),3)}")